# Language-Model Adapter — DIMER Artifact Inference Tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/language-model-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/language-model-pipeline/blob/main/tutorials/language_model_artifact_inference_colab.ipynb)

**Profile:** `ARTIFACT-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification `1.0`

This notebook consumes an **externally supplied PEFT adapter ZIP**. It safely validates the archive, resolves its immutable model identity through the canonical registry, checks producer/consumer runtime compatibility, reconstructs the adapter through the production `finetuner.inference` path, accepts a new user prompt, and exports machine-readable predictions.

**No training or fine-tuning occurs here.**

**Trust boundary.** Archive safety and manifest hashes establish internal consistency, not sender authenticity. A whole-archive SHA-256 only helps when it arrives through an independently trusted channel. The artifact uses safetensors/JSON and `trustRemoteCode=false`; do not treat path-safe extraction as permission to deserialize arbitrary executable formats.


## Prerequisites

- Upload exactly one adapter ZIP produced by the E2E notebook/production artifact contract.
- Use a CUDA-capable Colab/Jupyter runtime.
- The exact base model revision is downloaded from the model host because the adapter stores deltas, not a duplicate base model.
- `CUSTOM_PROMPT` remains local to the notebook runtime and is not sent to an external inference API.

The notebook records the candidate notebook revision separately from the immutable pipeline-runtime and finetuner-runtime revisions.


## 1. Install exact dependencies and immutable production source

The same `language-model-pipeline` support revision and `language-model-finetuner` production revision used by the E2E notebook are loaded here. The checkout SHA is verified before artifact handling.


In [ ]:
%pip -q install transformers==5.16.1 tokenizers==0.23.2 huggingface-hub==1.30.0 peft==0.20.0 accelerate==1.14.0 bitsandbytes==0.49.0 safetensors==0.8.0 datasets==4.8.5 pandas==2.3.3 PyYAML==6.0.3 Jinja2==3.1.6
%pip -q install --no-deps git+https://github.com/kurtvalcorza/language-model-pipeline.git@8a9935c20f90d90f333ce2a191eedb001a1f0830
!rm -rf /content/language-model-finetuner
!git clone -q https://github.com/kurtvalcorza/language-model-finetuner.git /content/language-model-finetuner
!git -C /content/language-model-finetuner checkout -q 4db4338b28db2260060ca3642fd29e3ca5e19119


In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
import torch

FINETUNER_ROOT = Path("/content/language-model-finetuner")
sys.path.insert(0, str(FINETUNER_ROOT / "src"))

from lmpipeline.tutorial_api import (
    assert_finetuner_checkout,
    assert_runtime_compatible,
    assert_tutorial_runtime,
    consume_adapter_archive,
    resolve_artifact_model,
    sha256_file,
    validate_prompt,
)
from finetuner.inference import (
    generate_reply,
    load_adapter_for_inference,
    verify_adapter_active,
)

PIPELINE_RUNTIME_REVISION = "8a9935c20f90d90f333ce2a191eedb001a1f0830"
FINETUNER_RUNTIME_REVISION = "4db4338b28db2260060ca3642fd29e3ca5e19119"
RUNTIME = assert_tutorial_runtime()
assert_finetuner_checkout(FINETUNER_ROOT, FINETUNER_RUNTIME_REVISION)
if not torch.cuda.is_available():
    raise RuntimeError("Artifact inference requires a CUDA GPU.")
print(json.dumps({
    "pipelineRuntimeRevision": PIPELINE_RUNTIME_REVISION,
    "finetunerRuntimeRevision": FINETUNER_RUNTIME_REVISION,
    "runtime": RUNTIME,
}, indent=2))


## 2. Upload and validate the external adapter ZIP

The ZIP comes from outside this notebook execution. Before any model state is loaded, `lmpipeline.tutorial_api.consume_adapter_archive` rejects path traversal, backslash-ambiguous paths, symlinks, duplicate members, excessive expanded size, manifest mismatches, missing required PEFT files, unexpected files, and invalid immutable revision provenance.


In [ ]:
EXPECTED_ARTIFACT_ZIP_SHA256 = "" # @param {type:"string"}

from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one adapter ZIP")
name, payload = next(iter(uploaded.items()))
if not name.lower().endswith(".zip"):
    raise ValueError("The external artifact must be a .zip archive")
ARCHIVE = Path("/content") / Path(name).name
ARCHIVE.write_bytes(payload)

ARTIFACT_ROOT, MANIFEST, PROVENANCE = consume_adapter_archive(
    ARCHIVE,
    extraction_root="/content/dimer-language-model-artifact",
    expected_archive_sha256=EXPECTED_ARTIFACT_ZIP_SHA256,
)
print({
    "archive": ARCHIVE.name,
    "archiveSha256": sha256_file(ARCHIVE),
    "manifestedFiles": len(MANIFEST["files"]),
    "totalBytes": MANIFEST["totalBytes"],
})


## 3. Verify canonical model identity and runtime-source compatibility

The artifact must name the same immutable base model/revision as the current canonical registry. Critical ML package versions must match. For artifacts created by the current E2E tutorial, the recorded pipeline and finetuner runtime-source SHAs must also equal the immutable revisions loaded above.

This distinguishes **notebook/PR head** from **runtime code identity** instead of pretending they are the same commit.


In [ ]:
ENTRY = resolve_artifact_model(PROVENANCE)
assert_runtime_compatible(PROVENANCE, RUNTIME)

RECORDED_RUNTIME_REVISIONS = PROVENANCE.get("runtimeRevisions") or {}
if RECORDED_RUNTIME_REVISIONS.get("pipeline") != PIPELINE_RUNTIME_REVISION:
    raise ValueError("Artifact was produced with a different pipeline runtime revision")
if RECORDED_RUNTIME_REVISIONS.get("finetuner") != FINETUNER_RUNTIME_REVISION:
    raise ValueError("Artifact was produced with a different finetuner runtime revision")

print(json.dumps({
    "modelKey": ENTRY.key,
    "baseModel": ENTRY.model_id,
    "baseModelRevision": ENTRY.revision,
    "datasetDigest": PROVENANCE.get("datasetDigest"),
    "packageVersions": PROVENANCE.get("packageVersions"),
    "runtimeRevisions": RECORDED_RUNTIME_REVISIONS,
}, indent=2))


## 4. Reconstruct through the production inference surface

`finetuner.inference.load_adapter_for_inference` reconstructs the tokenizer, production base-model loader, exact pinned base revision, and PEFT adapter. `verify_adapter_active` then proves the serialized adapter has non-zero LoRA B matrices and changes logits relative to adapter-off on the same reconstructed model.


In [ ]:
MODEL, TOKENIZER = load_adapter_for_inference(
    ARTIFACT_ROOT,
    entry=ENTRY,
    device="cuda",
    quantized=True,
)
ACTIVITY = verify_adapter_active(
    MODEL,
    TOKENIZER,
    prompt="Kumusta.",
)
print({
    "baseModel": ENTRY.model_id,
    "revision": ENTRY.revision,
    "adapterActivity": ACTIVITY,
    "gpuMemoryGiB": round(torch.cuda.memory_allocated() / 1024**3, 2),
})


## 5. Validate real new user input

Edit `CUSTOM_PROMPT`. The production inference renderer is used to count the rendered context before generation. The prompt plus requested generation budget must fit within the training-time sequence ceiling recorded in provenance.


In [ ]:
CUSTOM_PROMPT = "Sumulat ng dalawang pangungusap tungkol sa responsableng paggamit ng AI." # @param {type:"string"}
MAX_NEW_TOKENS = 128 # @param {type:"integer"}

TRAINING = (PROVENANCE.get("job") or {}).get("training") or {}
MAX_SEQUENCE_LENGTH = int(TRAINING.get("maxSequenceLength", 0))
if MAX_SEQUENCE_LENGTH <= 0 or MAX_SEQUENCE_LENGTH > int(ENTRY.max_sequence_length):
    raise ValueError("Artifact provenance has an invalid maxSequenceLength")
PROMPT_TOKENS = validate_prompt(
    TOKENIZER,
    CUSTOM_PROMPT,
    max_sequence_length=MAX_SEQUENCE_LENGTH,
)
if MAX_NEW_TOKENS <= 0 or PROMPT_TOKENS + MAX_NEW_TOKENS > MAX_SEQUENCE_LENGTH:
    raise ValueError(
        f"prompt ({PROMPT_TOKENS}) + MAX_NEW_TOKENS ({MAX_NEW_TOKENS}) exceeds "
        f"the {MAX_SEQUENCE_LENGTH}-token context ceiling"
    )
print({
    "promptTokens": PROMPT_TOKENS,
    "maxNewTokens": MAX_NEW_TOKENS,
    "contextCeiling": MAX_SEQUENCE_LENGTH,
})


## 6. Generate through production code and export machine-readable results

`finetuner.inference.generate_reply` is the shared generation implementation. The prediction is written to JSONL and a companion provenance JSON records the artifact SHA, base identity, decoding policy, package/runtime identity, and both runtime-source revisions.


In [ ]:
DECODING = {"do_sample": False}
ANSWER = generate_reply(
    MODEL,
    TOKENIZER,
    CUSTOM_PROMPT,
    max_new_tokens=MAX_NEW_TOKENS,
    decoding=DECODING,
)
if not ANSWER:
    raise RuntimeError("The reconstructed adapter generated an empty response")

RESULT = {
    "inputId": "prompt-1",
    "prompt": CUSTOM_PROMPT,
    "promptTokens": PROMPT_TOKENS,
    "output": ANSWER,
    "modelId": ENTRY.model_id,
    "modelRevision": ENTRY.revision,
    "decoding": {"doSample": False, "maxNewTokens": MAX_NEW_TOKENS},
}
OUTPUT_JSONL = Path("/content/artifact_inference_predictions.jsonl")
OUTPUT_JSONL.write_text(
    json.dumps(RESULT, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
OUTPUT_PROVENANCE = Path("/content/artifact_inference_provenance.json")
OUTPUT_PROVENANCE.write_text(
    json.dumps({
        "profile": "ARTIFACT-INFERENCE",
        "notebookSpecVersion": "1.0",
        "artifactSha256": sha256_file(ARCHIVE),
        "baseModel": ENTRY.model_id,
        "baseModelRevision": ENTRY.revision,
        "pipelineRuntimeRevision": PIPELINE_RUNTIME_REVISION,
        "finetunerRuntimeRevision": FINETUNER_RUNTIME_REVISION,
        "runtime": RUNTIME,
        "adapterActivity": ACTIVITY,
        "decoding": RESULT["decoding"],
    }, indent=2),
    encoding="utf-8",
)
display(pd.DataFrame([RESULT])[["inputId", "prompt", "output"]])
print("wrote", OUTPUT_JSONL, "and", OUTPUT_PROVENANCE)


## 7. Optional stochastic decoding

Greedy decoding is the reproducible verification path. Sampling is optional and must be explicit because it changes outputs and is not a confidence measure.


In [ ]:
RUN_SAMPLING_EXAMPLE = False # @param {type:"boolean"}
if RUN_SAMPLING_EXAMPLE:
    sampled = generate_reply(
        MODEL,
        TOKENIZER,
        CUSTOM_PROMPT,
        max_new_tokens=MAX_NEW_TOKENS,
        decoding={
            "do_sample": True,
            "temperature": 0.7,
            "top_p": 0.8,
            "top_k": 20,
        },
    )
    print(sampled)


## Interpretation and limits

A successful run establishes that the external artifact passed archive/manifest checks, its model/revision and runtime-source identities match the expected production code, the adapter reconstructed through the production inference surface, the reloaded LoRA deltas are non-zero and alter model logits, a real new prompt fit the declared context contract, and machine-readable outputs were written.

It does **not** establish sender authenticity unless the artifact/digest was independently authenticated, nor benchmark accuracy, factual correctness, safety, fairness, calibration, robustness, or production fitness. Release-grade status additionally requires a clean supported-runtime execution record tied to the candidate notebook/PR head and the two immutable runtime-source revisions.
